# SQL-Based ML Evaluation

This notebook loads the artifacts generated by the previous machine-learning notebook and evaluates the experiment using **SQLite and SQL**.

## Input files

- `synthetic_customer_churn.csv`
- `logistic_regression_predictions.csv`
- `logistic_regression_metrics.json`

The trained `.joblib` model is not inserted into SQL. It remains a reproducibility artifact.

## Main objectives

1. Create relational tables for customers, experiments, predictions, and reported metrics.
2. Validate data quality and consistency with SQL.
3. Recompute classification metrics independently with SQL.
4. Compare reported metrics against SQL-recomputed metrics.
5. Investigate false positives, false negatives, subgroups, and decision thresholds.
6. Export an SQLite database and analytical reports.


## 1. Import the required libraries

In [ ]:
import json
import sqlite3
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:.6f}")


## 2. Locate or upload the input files

When the notebook runs in Google Colab, it asks you to upload any missing input files.  
When the files are already in the working directory, no upload is necessary.


In [ ]:
REQUIRED_FILES = [
    "synthetic_customer_churn.csv",
    "logistic_regression_predictions.csv",
    "logistic_regression_metrics.json",
]

missing_files = [
    file_name
    for file_name in REQUIRED_FILES
    if not Path(file_name).exists()
]

if missing_files:
    try:
        from google.colab import files

        print("Upload the following files:")
        for file_name in missing_files:
            print("-", file_name)

        uploaded = files.upload()
    except ImportError as error:
        raise FileNotFoundError(
            "The required files are missing from the current directory: "
            + ", ".join(missing_files)
        ) from error

remaining_missing_files = [
    file_name
    for file_name in REQUIRED_FILES
    if not Path(file_name).exists()
]

if remaining_missing_files:
    raise FileNotFoundError(
        "These files are still missing: "
        + ", ".join(remaining_missing_files)
    )

print("All required files are available.")


Upload the following files:
- synthetic_customer_churn.csv
- logistic_regression_predictions.csv
- logistic_regression_metrics.json


Saving logistic_regression_metrics.json to logistic_regression_metrics.json
Saving logistic_regression_predictions.csv to logistic_regression_predictions.csv
Saving synthetic_customer_churn.csv to synthetic_customer_churn.csv
All required files are available.


## 3. Load the customer dataset, predictions, and reported metrics

In [ ]:
customers_df = pd.read_csv(
    "synthetic_customer_churn.csv"
)

predictions_df = pd.read_csv(
    "logistic_regression_predictions.csv"
)

with open(
    "logistic_regression_metrics.json",
    "r",
    encoding="utf-8",
) as file:
    reported_metrics = json.load(file)

print("Customer dataset shape:", customers_df.shape)
print("Prediction dataset shape:", predictions_df.shape)
print("Reported metric names:", list(reported_metrics))


Customer dataset shape: (5000, 10)
Prediction dataset shape: (1000, 8)
Reported metric names: ['accuracy', 'precision', 'recall', 'f1_score', 'roc_auc', 'pr_auc', 'log_loss']


## 4. Inspect the loaded files

In [ ]:
display(customers_df.head())
display(predictions_df.head())

reported_metrics_df = pd.DataFrame(
    [
        {
            "experiment_id": "EXP_001",
            "metric_name": metric_name,
            "metric_value": float(metric_value),
        }
        for metric_name, metric_value in reported_metrics.items()
    ]
)

display(reported_metrics_df)

,customer_id,age,region,plan,monthly_fee,months_active,usage_hours,support_tickets,failed_payments,churn
0,CUST_000001,45,South,Basic,35.720000,12,53.670000,1,1,1
1,CUST_000002,27,East,Basic,31.380000,20,23.320000,1,1,0
2,CUST_000003,51,East,Basic,35.730000,10,7.280000,2,0,0
3,CUST_000004,53,East,Basic,22.510000,24,43.790000,2,0,0
4,CUST_000005,18,North,Basic,27.740000,12,27.240000,2,0,0


,customer_id,y_true,y_pred,churn_probability,model_name,model_version,experiment_id,error_type
0,CUST_001104,0,0,0.278493,logistic_regression,v1,EXP_001,correct
1,CUST_000538,0,0,0.138242,logistic_regression,v1,EXP_001,correct
2,CUST_003397,0,0,0.322716,logistic_regression,v1,EXP_001,correct
3,CUST_001694,0,0,0.373723,logistic_regression,v1,EXP_001,correct
4,CUST_004514,0,0,0.358443,logistic_regression,v1,EXP_001,correct


,experiment_id,metric_name,metric_value
0,EXP_001,accuracy,0.665000
1,EXP_001,precision,0.300518
2,EXP_001,recall,0.640884
3,EXP_001,f1_score,0.409171
4,EXP_001,roc_auc,0.714859
5,EXP_001,pr_auc,0.408750
6,EXP_001,log_loss,0.619833


## 5. Perform basic Python checks before database insertion

In [ ]:
required_customer_columns = {
    "customer_id",
    "age",
    "region",
    "plan",
    "monthly_fee",
    "months_active",
    "usage_hours",
    "support_tickets",
    "failed_payments",
    "churn",
}

required_prediction_columns = {
    "customer_id",
    "y_true",
    "y_pred",
    "churn_probability",
    "model_name",
    "model_version",
    "experiment_id",
    "error_type",
}

missing_customer_columns = (
    required_customer_columns
    - set(customers_df.columns)
)

missing_prediction_columns = (
    required_prediction_columns
    - set(predictions_df.columns)
)

assert not missing_customer_columns, (
    f"Missing customer columns: {missing_customer_columns}"
)

assert not missing_prediction_columns, (
    f"Missing prediction columns: {missing_prediction_columns}"
)

print("The required columns are available.")


The required columns are available.


## 6. Create the SQLite database connection

In [ ]:
DATABASE_FILE = Path("ml_evaluation.db")

if DATABASE_FILE.exists():
    DATABASE_FILE.unlink()

connection = sqlite3.connect(
    DATABASE_FILE
)

connection.execute(
    "PRAGMA foreign_keys = ON"
)

print("Connected to:", DATABASE_FILE)


Connected to: ml_evaluation.db


## 7. Create the relational schema

In [ ]:
schema_sql = '''
CREATE TABLE customers (
    customer_id TEXT PRIMARY KEY,
    age INTEGER NOT NULL CHECK (age BETWEEN 18 AND 80),
    region TEXT NOT NULL,
    plan TEXT NOT NULL,
    monthly_fee REAL NOT NULL CHECK (monthly_fee >= 0),
    months_active INTEGER NOT NULL CHECK (months_active >= 0),
    usage_hours REAL NOT NULL CHECK (usage_hours >= 0),
    support_tickets INTEGER NOT NULL CHECK (support_tickets >= 0),
    failed_payments INTEGER NOT NULL CHECK (failed_payments >= 0),
    churn INTEGER NOT NULL CHECK (churn IN (0, 1))
);

CREATE TABLE experiments (
    experiment_id TEXT PRIMARY KEY,
    model_name TEXT NOT NULL,
    model_version TEXT NOT NULL,
    dataset_name TEXT NOT NULL,
    dataset_version TEXT NOT NULL,
    status TEXT NOT NULL CHECK (
        status IN ('completed', 'failed', 'invalid', 'warning')
    )
);

CREATE TABLE predictions (
    experiment_id TEXT NOT NULL,
    customer_id TEXT NOT NULL,
    y_true INTEGER NOT NULL CHECK (y_true IN (0, 1)),
    y_pred INTEGER NOT NULL CHECK (y_pred IN (0, 1)),
    churn_probability REAL NOT NULL CHECK (
        churn_probability BETWEEN 0 AND 1
    ),
    model_name TEXT NOT NULL,
    model_version TEXT NOT NULL,
    error_type TEXT NOT NULL,
    PRIMARY KEY (experiment_id, customer_id),
    FOREIGN KEY (experiment_id)
        REFERENCES experiments(experiment_id),
    FOREIGN KEY (customer_id)
        REFERENCES customers(customer_id)
);

CREATE TABLE reported_metrics (
    experiment_id TEXT NOT NULL,
    metric_name TEXT NOT NULL,
    metric_value REAL NOT NULL,
    PRIMARY KEY (experiment_id, metric_name),
    FOREIGN KEY (experiment_id)
        REFERENCES experiments(experiment_id)
);
'''

connection.executescript(schema_sql)
connection.commit()

print("Relational schema created.")


Relational schema created.


## 8. Insert the experiment metadata

In [ ]:
experiment_record = (
    "EXP_001",
    "logistic_regression",
    "v1",
    "synthetic_customer_churn",
    "v1",
    "completed",
)

connection.execute(
    '''
    INSERT INTO experiments (
        experiment_id,
        model_name,
        model_version,
        dataset_name,
        dataset_version,
        status
    )
    VALUES (?, ?, ?, ?, ?, ?)
    ''',
    experiment_record,
)

connection.commit()

pd.read_sql_query(
    "SELECT * FROM experiments",
    connection,
)


,experiment_id,model_name,model_version,dataset_name,dataset_version,status
0,EXP_001,logistic_regression,v1,synthetic_customer_churn,v1,completed


## 9. Insert the customer records

In [ ]:
customer_columns = [
    "customer_id",
    "age",
    "region",
    "plan",
    "monthly_fee",
    "months_active",
    "usage_hours",
    "support_tickets",
    "failed_payments",
    "churn",
]

customer_records = list(
    customers_df[customer_columns]
    .itertuples(index=False, name=None)
)

connection.executemany(
    '''
    INSERT INTO customers (
        customer_id,
        age,
        region,
        plan,
        monthly_fee,
        months_active,
        usage_hours,
        support_tickets,
        failed_payments,
        churn
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''',
    customer_records,
)

connection.commit()

pd.read_sql_query(
    "SELECT COUNT(*) AS customer_count FROM customers",
    connection,
)


,customer_count
0,5000


## 10. Insert the prediction records

In [ ]:
prediction_columns = [
    "experiment_id",
    "customer_id",
    "y_true",
    "y_pred",
    "churn_probability",
    "model_name",
    "model_version",
    "error_type",
]

prediction_records = list(
    predictions_df[prediction_columns]
    .itertuples(index=False, name=None)
)

connection.executemany(
    '''
    INSERT INTO predictions (
        experiment_id,
        customer_id,
        y_true,
        y_pred,
        churn_probability,
        model_name,
        model_version,
        error_type
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
    ''',
    prediction_records,
)

connection.commit()

pd.read_sql_query(
    "SELECT COUNT(*) AS prediction_count FROM predictions",
    connection,
)


,prediction_count
0,1000


## 11. Insert the reported metrics

In [ ]:
metric_records = list(
    reported_metrics_df[
        [
            "experiment_id",
            "metric_name",
            "metric_value",
        ]
    ].itertuples(index=False, name=None)
)

connection.executemany(
    '''
    INSERT INTO reported_metrics (
        experiment_id,
        metric_name,
        metric_value
    )
    VALUES (?, ?, ?)
    ''',
    metric_records,
)

connection.commit()

pd.read_sql_query(
    '''
    SELECT *
    FROM reported_metrics
    ORDER BY metric_name
    ''',
    connection,
)


,experiment_id,metric_name,metric_value
0,EXP_001,accuracy,0.665000
1,EXP_001,f1_score,0.409171
2,EXP_001,log_loss,0.619833
3,EXP_001,pr_auc,0.408750
4,EXP_001,precision,0.300518
5,EXP_001,recall,0.640884
6,EXP_001,roc_auc,0.714859


# SQL Data-Quality Validation

## 12. Count records in every table

In [ ]:
table_count_query = '''
SELECT
    'customers' AS table_name,
    COUNT(*) AS record_count
FROM customers

UNION ALL

SELECT
    'experiments' AS table_name,
    COUNT(*) AS record_count
FROM experiments

UNION ALL

SELECT
    'predictions' AS table_name,
    COUNT(*) AS record_count
FROM predictions

UNION ALL

SELECT
    'reported_metrics' AS table_name,
    COUNT(*) AS record_count
FROM reported_metrics;
'''

table_counts = pd.read_sql_query(
    table_count_query,
    connection,
)

table_counts


,table_name,record_count
0,customers,5000
1,experiments,1
2,predictions,1000
3,reported_metrics,7


## 13. Check for duplicated prediction keys

In [ ]:
duplicate_predictions_query = '''
SELECT
    experiment_id,
    customer_id,
    COUNT(*) AS prediction_count
FROM predictions
GROUP BY
    experiment_id,
    customer_id
HAVING COUNT(*) > 1;
'''

duplicate_predictions = pd.read_sql_query(
    duplicate_predictions_query,
    connection,
)

print("Duplicated prediction keys:", len(duplicate_predictions))
duplicate_predictions


Duplicated prediction keys: 0


,experiment_id,customer_id,prediction_count


## 14. Check for invalid or missing probabilities

In [ ]:
invalid_probability_query = '''
SELECT *
FROM predictions
WHERE churn_probability IS NULL
   OR churn_probability < 0
   OR churn_probability > 1;
'''

invalid_probabilities = pd.read_sql_query(
    invalid_probability_query,
    connection,
)

print("Invalid probabilities:", len(invalid_probabilities))
invalid_probabilities


Invalid probabilities: 0


,experiment_id,customer_id,y_true,y_pred,churn_probability,model_name,model_version,error_type


## 15. Check for invalid labels

In [ ]:
invalid_label_query = '''
SELECT *
FROM predictions
WHERE y_true NOT IN (0, 1)
   OR y_pred NOT IN (0, 1)
   OR y_true IS NULL
   OR y_pred IS NULL;
'''

invalid_labels = pd.read_sql_query(
    invalid_label_query,
    connection,
)

print("Invalid labels:", len(invalid_labels))
invalid_labels


Invalid labels: 0


,experiment_id,customer_id,y_true,y_pred,churn_probability,model_name,model_version,error_type


## 16. Check for predictions without matching customers

In [ ]:
orphan_prediction_query = '''
SELECT
    p.experiment_id,
    p.customer_id
FROM predictions AS p
LEFT JOIN customers AS c
    ON p.customer_id = c.customer_id
WHERE c.customer_id IS NULL;
'''

orphan_predictions = pd.read_sql_query(
    orphan_prediction_query,
    connection,
)

print("Orphan predictions:", len(orphan_predictions))
orphan_predictions


Orphan predictions: 0


,experiment_id,customer_id


## 17. Compare prediction labels with the source customer labels

In [ ]:
label_consistency_query = '''
SELECT
    p.customer_id,
    p.y_true AS prediction_label,
    c.churn AS customer_table_label
FROM predictions AS p
INNER JOIN customers AS c
    ON p.customer_id = c.customer_id
WHERE p.y_true <> c.churn;
'''

label_inconsistencies = pd.read_sql_query(
    label_consistency_query,
    connection,
)

print("Inconsistent true labels:", len(label_inconsistencies))
label_inconsistencies


Inconsistent true labels: 0


,customer_id,prediction_label,customer_table_label


## 18. Check model metadata consistency

In [ ]:
metadata_consistency_query = '''
SELECT
    p.experiment_id,
    p.model_name AS prediction_model_name,
    e.model_name AS experiment_model_name,
    p.model_version AS prediction_model_version,
    e.model_version AS experiment_model_version,
    COUNT(*) AS affected_rows
FROM predictions AS p
INNER JOIN experiments AS e
    ON p.experiment_id = e.experiment_id
WHERE p.model_name <> e.model_name
   OR p.model_version <> e.model_version
GROUP BY
    p.experiment_id,
    p.model_name,
    e.model_name,
    p.model_version,
    e.model_version;
'''

metadata_inconsistencies = pd.read_sql_query(
    metadata_consistency_query,
    connection,
)

print(
    "Metadata inconsistency groups:",
    len(metadata_inconsistencies),
)

metadata_inconsistencies


Metadata inconsistency groups: 0


,experiment_id,prediction_model_name,experiment_model_name,prediction_model_version,experiment_model_version,affected_rows


## 19. Produce a compact data-quality summary

In [ ]:
data_quality_summary = pd.DataFrame(
    {
        "check_name": [
            "duplicated_prediction_keys",
            "invalid_probabilities",
            "invalid_labels",
            "orphan_predictions",
            "inconsistent_true_labels",
            "metadata_inconsistencies",
        ],
        "issue_count": [
            len(duplicate_predictions),
            len(invalid_probabilities),
            len(invalid_labels),
            len(orphan_predictions),
            len(label_inconsistencies),
            len(metadata_inconsistencies),
        ],
    }
)

data_quality_summary["status"] = np.where(
    data_quality_summary["issue_count"] == 0,
    "passed",
    "failed",
)

data_quality_summary


,check_name,issue_count,status
0,duplicated_prediction_keys,0,passed
1,invalid_probabilities,0,passed
2,invalid_labels,0,passed
3,orphan_predictions,0,passed
4,inconsistent_true_labels,0,passed
5,metadata_inconsistencies,0,passed


# SQL Metric Recalculation

## 20. Recompute the confusion-matrix components

In [ ]:
confusion_query = '''
SELECT
    experiment_id,

    SUM(
        CASE
            WHEN y_true = 1 AND y_pred = 1
            THEN 1
            ELSE 0
        END
    ) AS true_positive,

    SUM(
        CASE
            WHEN y_true = 0 AND y_pred = 0
            THEN 1
            ELSE 0
        END
    ) AS true_negative,

    SUM(
        CASE
            WHEN y_true = 0 AND y_pred = 1
            THEN 1
            ELSE 0
        END
    ) AS false_positive,

    SUM(
        CASE
            WHEN y_true = 1 AND y_pred = 0
            THEN 1
            ELSE 0
        END
    ) AS false_negative

FROM predictions
GROUP BY experiment_id;
'''

confusion_sql = pd.read_sql_query(
    confusion_query,
    connection,
)

confusion_sql


,experiment_id,true_positive,true_negative,false_positive,false_negative
0,EXP_001,116,549,270,65


## 21. Recompute accuracy, precision, recall, specificity, and F1-score

In [ ]:
classification_metrics_query = '''
WITH confusion AS (
    SELECT
        experiment_id,

        SUM(
            CASE
                WHEN y_true = 1 AND y_pred = 1
                THEN 1
                ELSE 0
            END
        ) AS tp,

        SUM(
            CASE
                WHEN y_true = 0 AND y_pred = 0
                THEN 1
                ELSE 0
            END
        ) AS tn,

        SUM(
            CASE
                WHEN y_true = 0 AND y_pred = 1
                THEN 1
                ELSE 0
            END
        ) AS fp,

        SUM(
            CASE
                WHEN y_true = 1 AND y_pred = 0
                THEN 1
                ELSE 0
            END
        ) AS fn

    FROM predictions
    GROUP BY experiment_id
),

base_metrics AS (
    SELECT
        experiment_id,
        (tp + tn) * 1.0
            / NULLIF(tp + tn + fp + fn, 0)
            AS accuracy,

        tp * 1.0
            / NULLIF(tp + fp, 0)
            AS precision,

        tp * 1.0
            / NULLIF(tp + fn, 0)
            AS recall,

        tn * 1.0
            / NULLIF(tn + fp, 0)
            AS specificity
    FROM confusion
)

SELECT
    experiment_id,
    accuracy,
    precision,
    recall,
    specificity,

    2.0 * precision * recall
        / NULLIF(precision + recall, 0)
        AS f1_score

FROM base_metrics;
'''

classification_metrics_sql = pd.read_sql_query(
    classification_metrics_query,
    connection,
)

classification_metrics_sql


,experiment_id,accuracy,precision,recall,specificity,f1_score
0,EXP_001,0.665000,0.300518,0.640884,0.670330,0.409171


## 22. Recompute ROC-AUC using ranks in SQL

In [ ]:
roc_auc_query = '''
WITH ranked_predictions AS (
    SELECT
        experiment_id,
        customer_id,
        y_true,
        churn_probability,

        RANK() OVER (
            PARTITION BY experiment_id
            ORDER BY churn_probability ASC
        ) AS base_rank,

        COUNT(*) OVER (
            PARTITION BY
                experiment_id,
                churn_probability
        ) AS tie_count

    FROM predictions
),

rank_summary AS (
    SELECT
        experiment_id,

        SUM(
            CASE
                WHEN y_true = 1
                THEN base_rank + (tie_count - 1) / 2.0
                ELSE 0
            END
        ) AS positive_rank_sum,

        SUM(
            CASE
                WHEN y_true = 1
                THEN 1
                ELSE 0
            END
        ) AS positive_count,

        SUM(
            CASE
                WHEN y_true = 0
                THEN 1
                ELSE 0
            END
        ) AS negative_count

    FROM ranked_predictions
    GROUP BY experiment_id
)

SELECT
    experiment_id,

    (
        positive_rank_sum
        - positive_count * (positive_count + 1) / 2.0
    )
    / NULLIF(
        positive_count * negative_count,
        0
    ) AS roc_auc

FROM rank_summary;
'''

roc_auc_sql = pd.read_sql_query(
    roc_auc_query,
    connection,
)

roc_auc_sql


,experiment_id,roc_auc
0,EXP_001,0.714859


## 23. Recompute average precision (PR-AUC summary metric) in SQL

In [ ]:
average_precision_query = '''
WITH ordered_predictions AS (
    SELECT
        experiment_id,
        customer_id,
        y_true,
        churn_probability,

        ROW_NUMBER() OVER (
            PARTITION BY experiment_id
            ORDER BY
                churn_probability DESC,
                customer_id ASC
        ) AS prediction_rank

    FROM predictions
),

running_counts AS (
    SELECT
        experiment_id,
        customer_id,
        y_true,
        churn_probability,
        prediction_rank,

        SUM(y_true) OVER (
            PARTITION BY experiment_id
            ORDER BY prediction_rank
            ROWS BETWEEN
                UNBOUNDED PRECEDING
                AND CURRENT ROW
        ) AS cumulative_true_positives

    FROM ordered_predictions
),

experiment_counts AS (
    SELECT
        experiment_id,
        SUM(y_true) AS positive_count
    FROM predictions
    GROUP BY experiment_id
)

SELECT
    r.experiment_id,

    SUM(
        CASE
            WHEN r.y_true = 1
            THEN (
                r.cumulative_true_positives * 1.0
                / r.prediction_rank
            )
            ELSE 0
        END
    )
    / NULLIF(e.positive_count, 0)
    AS pr_auc

FROM running_counts AS r

INNER JOIN experiment_counts AS e
    ON r.experiment_id = e.experiment_id

GROUP BY r.experiment_id;
'''

average_precision_sql = pd.read_sql_query(
    average_precision_query,
    connection,
)

average_precision_sql


,experiment_id,pr_auc
0,EXP_001,0.408750


## 24. Recompute log loss in SQL

In [ ]:
log_loss_query = '''
SELECT
    experiment_id,

    -AVG(
        y_true
        * LN(
            MAX(
                MIN(churn_probability, 0.999999999999),
                0.000000000001
            )
        )
        +
        (1 - y_true)
        * LN(
            1
            - MAX(
                MIN(churn_probability, 0.999999999999),
                0.000000000001
            )
        )
    ) AS log_loss

FROM predictions
GROUP BY experiment_id;
'''

log_loss_sql = pd.read_sql_query(
    log_loss_query,
    connection,
)

log_loss_sql


,experiment_id,log_loss
0,EXP_001,0.619833


## 25. Combine all SQL-recomputed metrics

In [ ]:
sql_metrics_wide = (
    classification_metrics_sql
    .merge(
        roc_auc_sql,
        on="experiment_id",
        how="left",
    )
    .merge(
        average_precision_sql,
        on="experiment_id",
        how="left",
    )
    .merge(
        log_loss_sql,
        on="experiment_id",
        how="left",
    )
)

sql_metrics_wide


,experiment_id,accuracy,precision,recall,specificity,f1_score,roc_auc,pr_auc,log_loss
0,EXP_001,0.665000,0.300518,0.640884,0.670330,0.409171,0.714859,0.408750,0.619833


## 26. Convert the SQL metrics to a long analytical table

In [ ]:
sql_metrics_long = sql_metrics_wide.melt(
    id_vars="experiment_id",
    var_name="metric_name",
    value_name="sql_recomputed_value",
)

sql_metrics_long


,experiment_id,metric_name,sql_recomputed_value
0,EXP_001,accuracy,0.665000
1,EXP_001,precision,0.300518
2,EXP_001,recall,0.640884
3,EXP_001,specificity,0.670330
4,EXP_001,f1_score,0.409171
5,EXP_001,roc_auc,0.714859
6,EXP_001,pr_auc,0.408750
7,EXP_001,log_loss,0.619833


## 27. Compare reported metrics against SQL-recomputed metrics

In [ ]:
metric_comparison = (
    reported_metrics_df
    .merge(
        sql_metrics_long,
        on=[
            "experiment_id",
            "metric_name",
        ],
        how="left",
    )
)

metric_comparison["absolute_difference"] = (
    metric_comparison["metric_value"]
    - metric_comparison["sql_recomputed_value"]
).abs()

METRIC_TOLERANCE = 1e-9

metric_comparison["validation_status"] = np.where(
    metric_comparison["absolute_difference"]
    <= METRIC_TOLERANCE,
    "matched",
    "mismatch",
)

metric_comparison = metric_comparison.rename(
    columns={
        "metric_value": "reported_value",
    }
)

metric_comparison


,experiment_id,metric_name,reported_value,sql_recomputed_value,absolute_difference,validation_status
0,EXP_001,accuracy,0.665000,0.665000,0.000000,matched
1,EXP_001,precision,0.300518,0.300518,0.000000,matched
2,EXP_001,recall,0.640884,0.640884,0.000000,matched
3,EXP_001,f1_score,0.409171,0.409171,0.000000,matched
4,EXP_001,roc_auc,0.714859,0.714859,0.000000,matched
5,EXP_001,pr_auc,0.408750,0.408750,0.000000,matched
6,EXP_001,log_loss,0.619833,0.619833,0.000000,matched


The SQL average-precision calculation orders predictions by probability and customer ID.  
For datasets with many exactly tied probabilities, a library implementation can handle interpolation or ties differently. This dataset normally has unique continuous probabilities, so the results should closely match.


# Error and Subgroup Analysis

## 28. Count correct predictions, false positives, and false negatives

In [ ]:
error_type_query = '''
SELECT
    error_type,
    COUNT(*) AS prediction_count,
    COUNT(*) * 100.0
        / SUM(COUNT(*)) OVER ()
        AS percentage
FROM predictions
GROUP BY error_type
ORDER BY prediction_count DESC;
'''

error_type_summary = pd.read_sql_query(
    error_type_query,
    connection,
)

error_type_summary


,error_type,prediction_count,percentage
0,correct,665,66.500000
1,false_positive,270,27.000000
2,false_negative,65,6.500000


## 29. Inspect the most confident incorrect predictions

In [ ]:
confident_errors_query = '''
SELECT
    p.customer_id,
    p.y_true,
    p.y_pred,
    p.churn_probability,
    p.error_type,

    CASE
        WHEN p.y_pred = 1
        THEN p.churn_probability
        ELSE 1 - p.churn_probability
    END AS prediction_confidence,

    c.age,
    c.region,
    c.plan,
    c.monthly_fee,
    c.months_active,
    c.usage_hours,
    c.support_tickets,
    c.failed_payments

FROM predictions AS p

INNER JOIN customers AS c
    ON p.customer_id = c.customer_id

WHERE p.y_true <> p.y_pred

ORDER BY prediction_confidence DESC
LIMIT 20;
'''

confident_errors = pd.read_sql_query(
    confident_errors_query,
    connection,
)

confident_errors


,customer_id,y_true,y_pred,churn_probability,error_type,prediction_confidence,age,region,plan,monthly_fee,months_active,usage_hours,support_tickets,failed_payments
0,CUST_003528,0,1,0.923950,false_positive,0.923950,24,East,Basic,35.010000,17,34.960000,2,3
1,CUST_002575,0,1,0.889269,false_positive,0.889269,40,East,Standard,63.360000,21,7.910000,2,2
2,CUST_002269,0,1,0.864985,false_positive,0.864985,18,South,Standard,47.970000,26,25.870000,2,2
3,CUST_001576,0,1,0.857668,false_positive,0.857668,25,South,Standard,75.900000,21,11.430000,2,1
4,CUST_003193,0,1,0.841566,false_positive,0.841566,42,South,Standard,69.260000,10,41.310000,2,2
5,CUST_003160,0,1,0.837169,false_positive,0.837169,32,East,Basic,31.830000,39,8.780000,2,2
6,CUST_000698,0,1,0.833723,false_positive,0.833723,28,South,Basic,34.290000,4,22.490000,2,1
7,CUST_001377,0,1,0.829508,false_positive,0.829508,48,East,Premium,95.870000,30,22.530000,4,1
8,CUST_004661,0,1,0.825022,false_positive,0.825022,20,East,Standard,69.860000,4,26.340000,1,1
9,CUST_000312,0,1,0.824823,false_positive,0.824823,47,West,Basic,38.610000,7,7.630000,2,1


## 30. Analyze model performance by subscription plan

In [ ]:
plan_performance_query = '''
WITH plan_confusion AS (
    SELECT
        c.plan,

        COUNT(*) AS sample_count,

        SUM(
            CASE
                WHEN p.y_true = 1 AND p.y_pred = 1
                THEN 1
                ELSE 0
            END
        ) AS tp,

        SUM(
            CASE
                WHEN p.y_true = 0 AND p.y_pred = 0
                THEN 1
                ELSE 0
            END
        ) AS tn,

        SUM(
            CASE
                WHEN p.y_true = 0 AND p.y_pred = 1
                THEN 1
                ELSE 0
            END
        ) AS fp,

        SUM(
            CASE
                WHEN p.y_true = 1 AND p.y_pred = 0
                THEN 1
                ELSE 0
            END
        ) AS fn

    FROM predictions AS p

    INNER JOIN customers AS c
        ON p.customer_id = c.customer_id

    GROUP BY c.plan
),

plan_metrics AS (
    SELECT
        plan,
        sample_count,
        tp,
        tn,
        fp,
        fn,

        (tp + tn) * 1.0
            / NULLIF(sample_count, 0)
            AS accuracy,

        tp * 1.0
            / NULLIF(tp + fp, 0)
            AS precision,

        tp * 1.0
            / NULLIF(tp + fn, 0)
            AS recall

    FROM plan_confusion
)

SELECT
    plan,
    sample_count,
    tp,
    tn,
    fp,
    fn,
    accuracy,
    precision,
    recall,

    2.0 * precision * recall
        / NULLIF(precision + recall, 0)
        AS f1_score

FROM plan_metrics
ORDER BY f1_score ASC;
'''

plan_performance = pd.read_sql_query(
    plan_performance_query,
    connection,
)

plan_performance


,plan,sample_count,tp,tn,fp,fn,accuracy,precision,recall,f1_score
0,Basic,423,26,262,106,29,0.680851,0.196970,0.472727,0.278075
1,Premium,172,35,59,68,10,0.546512,0.339806,0.777778,0.472973
2,Standard,405,55,228,96,26,0.698765,0.364238,0.679012,0.474138


## 31. Analyze model performance by region

In [ ]:
region_performance_query = '''
WITH region_confusion AS (
    SELECT
        c.region,

        COUNT(*) AS sample_count,

        SUM(
            CASE
                WHEN p.y_true = 1 AND p.y_pred = 1
                THEN 1
                ELSE 0
            END
        ) AS tp,

        SUM(
            CASE
                WHEN p.y_true = 0 AND p.y_pred = 0
                THEN 1
                ELSE 0
            END
        ) AS tn,

        SUM(
            CASE
                WHEN p.y_true = 0 AND p.y_pred = 1
                THEN 1
                ELSE 0
            END
        ) AS fp,

        SUM(
            CASE
                WHEN p.y_true = 1 AND p.y_pred = 0
                THEN 1
                ELSE 0
            END
        ) AS fn

    FROM predictions AS p

    INNER JOIN customers AS c
        ON p.customer_id = c.customer_id

    GROUP BY c.region
),

region_metrics AS (
    SELECT
        region,
        sample_count,
        tp,
        tn,
        fp,
        fn,

        (tp + tn) * 1.0
            / NULLIF(sample_count, 0)
            AS accuracy,

        tp * 1.0
            / NULLIF(tp + fp, 0)
            AS precision,

        tp * 1.0
            / NULLIF(tp + fn, 0)
            AS recall

    FROM region_confusion
)

SELECT
    region,
    sample_count,
    tp,
    tn,
    fp,
    fn,
    accuracy,
    precision,
    recall,

    2.0 * precision * recall
        / NULLIF(precision + recall, 0)
        AS f1_score

FROM region_metrics
ORDER BY f1_score ASC;
'''

region_performance = pd.read_sql_query(
    region_performance_query,
    connection,
)

region_performance


,region,sample_count,tp,tn,fp,fn,accuracy,precision,recall,f1_score
0,South,179,16,100,50,13,0.648045,0.242424,0.551724,0.336842
1,Central,157,18,77,49,13,0.605096,0.268657,0.580645,0.367347
2,West,256,33,132,79,12,0.644531,0.294643,0.733333,0.420382
3,East,248,30,136,66,16,0.669355,0.312500,0.652174,0.422535
4,North,160,19,104,26,11,0.768750,0.422222,0.633333,0.506667


## 32. Analyze errors by failed-payment count

In [ ]:
failed_payment_analysis_query = '''
SELECT
    c.failed_payments,
    COUNT(*) AS sample_count,

    AVG(p.y_true) AS observed_churn_rate,
    AVG(p.churn_probability) AS average_predicted_probability,

    AVG(
        CASE
            WHEN p.y_true = p.y_pred
            THEN 1.0
            ELSE 0.0
        END
    ) AS accuracy,

    SUM(
        CASE
            WHEN p.y_true = 1 AND p.y_pred = 0
            THEN 1
            ELSE 0
        END
    ) AS false_negatives,

    SUM(
        CASE
            WHEN p.y_true = 0 AND p.y_pred = 1
            THEN 1
            ELSE 0
        END
    ) AS false_positives

FROM predictions AS p

INNER JOIN customers AS c
    ON p.customer_id = c.customer_id

GROUP BY c.failed_payments
ORDER BY c.failed_payments;
'''

failed_payment_analysis = pd.read_sql_query(
    failed_payment_analysis_query,
    connection,
)

failed_payment_analysis


,failed_payments,sample_count,observed_churn_rate,average_predicted_probability,accuracy,false_negatives,false_positives
0,0,753,0.138114,0.411771,0.714475,57,158
1,1,208,0.264423,0.576910,0.485577,8,99
2,2,32,0.562500,0.761366,0.656250,0,11
3,3,7,0.571429,0.823201,0.714286,0,2


# Decision-Threshold Analysis

The stored `y_pred` values use the original model threshold.  
The next SQL query recalculates labels from probabilities at several thresholds and evaluates the trade-off between false positives and false negatives.


## 33. Compare multiple classification thresholds

In [ ]:
threshold_analysis_query = '''
WITH thresholds(threshold) AS (
    VALUES
        (0.20),
        (0.30),
        (0.40),
        (0.50),
        (0.60),
        (0.70),
        (0.80)
),

threshold_predictions AS (
    SELECT
        t.threshold,
        p.y_true,

        CASE
            WHEN p.churn_probability >= t.threshold
            THEN 1
            ELSE 0
        END AS threshold_prediction

    FROM predictions AS p
    CROSS JOIN thresholds AS t
),

threshold_confusion AS (
    SELECT
        threshold,

        SUM(
            CASE
                WHEN y_true = 1
                 AND threshold_prediction = 1
                THEN 1
                ELSE 0
            END
        ) AS tp,

        SUM(
            CASE
                WHEN y_true = 0
                 AND threshold_prediction = 0
                THEN 1
                ELSE 0
            END
        ) AS tn,

        SUM(
            CASE
                WHEN y_true = 0
                 AND threshold_prediction = 1
                THEN 1
                ELSE 0
            END
        ) AS fp,

        SUM(
            CASE
                WHEN y_true = 1
                 AND threshold_prediction = 0
                THEN 1
                ELSE 0
            END
        ) AS fn

    FROM threshold_predictions
    GROUP BY threshold
),

threshold_metrics AS (
    SELECT
        threshold,
        tp,
        tn,
        fp,
        fn,

        (tp + tn) * 1.0
            / NULLIF(tp + tn + fp + fn, 0)
            AS accuracy,

        tp * 1.0
            / NULLIF(tp + fp, 0)
            AS precision,

        tp * 1.0
            / NULLIF(tp + fn, 0)
            AS recall,

        (
            500 * fn
            + 30 * fp
        ) AS estimated_business_cost

    FROM threshold_confusion
)

SELECT
    threshold,
    tp,
    tn,
    fp,
    fn,
    accuracy,
    precision,
    recall,

    2.0 * precision * recall
        / NULLIF(precision + recall, 0)
        AS f1_score,

    estimated_business_cost

FROM threshold_metrics
ORDER BY threshold;
'''

threshold_analysis = pd.read_sql_query(
    threshold_analysis_query,
    connection,
)

threshold_analysis


,threshold,tp,tn,fp,fn,accuracy,precision,recall,f1_score,estimated_business_cost
0,0.200000,180,64,755,1,0.244000,0.192513,0.994475,0.322581,23150
1,0.300000,169,189,630,12,0.358000,0.211514,0.933702,0.344898,24900
2,0.400000,145,369,450,36,0.514000,0.243697,0.801105,0.373711,31500
3,0.500000,116,549,270,65,0.665000,0.300518,0.640884,0.409171,40600
4,0.600000,83,676,143,98,0.759000,0.367257,0.458564,0.407862,53290
5,0.700000,55,763,56,126,0.818000,0.495495,0.303867,0.376712,64680
6,0.800000,25,801,18,156,0.826000,0.581395,0.138122,0.223214,78540


## 34. Select the threshold with the lowest estimated business cost

In [ ]:
best_threshold = threshold_analysis.sort_values(
    [
        "estimated_business_cost",
        "threshold",
    ]
).head(1)

best_threshold


,threshold,tp,tn,fp,fn,accuracy,precision,recall,f1_score,estimated_business_cost
0,0.200000,180,64,755,1,0.244000,0.192513,0.994475,0.322581,23150


# Export Analytical Artifacts

## 35. Save SQL results as CSV reports

In [ ]:
DATA_QUALITY_REPORT = Path(
    "sql_data_quality_report.csv"
)

METRIC_VALIDATION_REPORT = Path(
    "sql_metric_validation.csv"
)

PLAN_REPORT = Path(
    "sql_plan_performance.csv"
)

REGION_REPORT = Path(
    "sql_region_performance.csv"
)

THRESHOLD_REPORT = Path(
    "sql_threshold_analysis.csv"
)

data_quality_summary.to_csv(
    DATA_QUALITY_REPORT,
    index=False,
)

metric_comparison.to_csv(
    METRIC_VALIDATION_REPORT,
    index=False,
)

plan_performance.to_csv(
    PLAN_REPORT,
    index=False,
)

region_performance.to_csv(
    REGION_REPORT,
    index=False,
)

threshold_analysis.to_csv(
    THRESHOLD_REPORT,
    index=False,
)

print("Saved:", DATABASE_FILE)
print("Saved:", DATA_QUALITY_REPORT)
print("Saved:", METRIC_VALIDATION_REPORT)
print("Saved:", PLAN_REPORT)
print("Saved:", REGION_REPORT)
print("Saved:", THRESHOLD_REPORT)


Saved: ml_evaluation.db
Saved: sql_data_quality_report.csv
Saved: sql_metric_validation.csv
Saved: sql_plan_performance.csv
Saved: sql_region_performance.csv
Saved: sql_threshold_analysis.csv


## 36. Run final assertions

In [ ]:
assert (
    data_quality_summary["issue_count"] == 0
).all()

assert (
    metric_comparison["validation_status"]
    == "matched"
).all()

assert DATABASE_FILE.exists()
assert METRIC_VALIDATION_REPORT.exists()

print("All final SQL evaluation checks passed.")


All final SQL evaluation checks passed.


## 37. Close the database connection

In [ ]:
connection.close()

print("Database connection closed.")


Database connection closed.


## 38. Optional: download the generated artifacts in Google Colab

In [ ]:
if "google.colab" in sys.modules:
    from google.colab import files

    files.download(str(DATABASE_FILE))
    files.download(str(DATA_QUALITY_REPORT))
    files.download(str(METRIC_VALIDATION_REPORT))
    files.download(str(PLAN_REPORT))
    files.download(str(REGION_REPORT))
    files.download(str(THRESHOLD_REPORT))
else:
    print(
        "The output files are available in the current working directory."
    )


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# What this notebook demonstrates

This workflow shows that you can:

- integrate Python and SQL;
- design relational tables for ML experiment outputs;
- validate data quality and referential consistency;
- independently recompute ML metrics;
- compare reported and recomputed results;
- analyze failure modes and subgroups;
- evaluate decision thresholds using business costs;
- produce reproducible analytical artifacts.
